# Homework 08

* Please follow the [guidelines](https://fmfi-compbio.github.io/viz/Tutorial1.html\#pokyny-k-zadaniam) for submitting notebooks. 
* Do not forget to switch off AI assistants in Colab `Menu->Tools->Settings->AI Assistance`.
* Use only the libraries imported in the cell below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # for making directories

## Importing data

Country indicators from World Bank, https://databank.worldbank.org/home under CC BY 4.0 license.

The table contains country population, surface area in km squared, GDP per capita (current US$), life expectancy at birth (years), fertility rate (births per woman); in years 2000, 2010, 2020.

We add a new column `GDPRelChange`, which is the change between `GDP2020` and `GDP2010` divided by `GDP2010`. For example, value of 0 means no change, value of 0.1 means 10% increase and -0.1 means 10% decrease.

As before, we will use only countries with names starting with S and having at least 3 million inhabitants in 2020. We also drop countries with missing values to avoid problems. Selected countries are in table `selection`.

In [ ]:
# read data
url = 'https://fmfi-compbio.github.io/viz/data/World_bank.csv'
countries = pd.read_csv(url).set_index('Country')

# new column GDPRelChange
countries['GDPRelChange'] = (countries['GDP2020'] - countries['GDP2010']) / countries['GDP2010']

# selecting countries
selection = countries.query('Population2020 > 3e6 and Country >= "Si" and Country <= "Sz"').dropna()

# printing new column for selected countries
print("GDPRelChange column:")
display(selection['GDPRelChange'])

## Three example plots with color palettes

Function `country_plot` draws a scatter plot using GDP in 2020 as x axis (in log scale), life expectancy in 2020 as y axis and a specified column as color. The name of the column is given in parameter `colorby`. Parameter `palette` is the color palette used in the plot. 

In [ ]:
def country_plot(axes, colorby, palette):
  # scatterplot itself
  sns.scatterplot(data=selection, x='GDP2020', y='Expectancy2020',
                  hue=colorby, s=200, ax=axes, palette=palette)

  # add labels for countries
  for i in range(selection.shape[0]):
    axes.text(x=selection['GDP2020'].iloc[i],
             y=selection['Expectancy2020'].iloc[i],
             s="  " + selection.index[i], size='small', color='black')
          
  # extend x range so that labels fit
  xmin = selection['GDP2020'].min()
  xmax = selection['GDP2020'].max()
  axes.set_xlim(xmin * 0.9, xmax * 2.6)
  # axes labels, log scale
  axes.set_xlabel('GDP per capita (US dollars), logscale')
  axes.set_ylabel('Life expectancy (years)')
  axes.semilogx()
  # legend outside the plot
  axes.legend(bbox_to_anchor=(1.05, 1), loc=2)


The code below creates three plots, each using a different column as `colorby` in `country_plot`. 

In [ ]:
# set up figure
figure, axes = plt.subplots(3, 1, figsize=(8,15), layout='tight')

# for categorical variable use the name of palette as string (here tab10)
country_plot(axes[0], 'Region', "tab10")
axes[0].set_title('Country indicators 2020, colored by region')

# for numerical variables use a color map created using sns.color_palette
# with as_cmap=True
colormap = sns.color_palette("flare", as_cmap=True)
country_plot(axes[1], 'Fertility2020', colormap)
axes[1].set_title('Country indicators 2020, colored by fertility')

country_plot(axes[2], 'GDPRelChange', colormap)
axes[2].set_title('Country indicators 2020, colored by relative GDP change from 2010')

pass

## Color palette tutorial

Seaborn has a [tutorial](https://seaborn.pydata.org/tutorial/color_palettes.html) describing various color palettes. You may need to read parts of it in this homework.

## Task 1a: Qualitative palettes [1 point]

Create three plots, each using `Region` as `colorby` argument in `country_plot`. In each plot, use a different palette. Include palettes `tab10`, `deep` and one additional [qualitative palette](https://seaborn.pydata.org/tutorial/color_palettes.html#qualitative-color-palettes) of your choice. Mention the palette names in the plot titles.

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8,15), layout='tight')

### TASK 1 START
# country_plot(axes[0],...)
# axes[0].set_title(...)
# ...
### TASK 1 END

pass

## Task 1b: qualitative palettes - discussion [1 point]

Answer the two questions below.

### TASK 1b START

(1) Explain what the [tutorial](https://seaborn.pydata.org/tutorial/color_palettes.html) claims are respective advantages of palettes `tab10` and `deep` (default Seaborn palette):


(2) Which of the three palettes used in Task 1a you like personally best?


### TASK 1b END

## Task 2a: quantitative palettes [2 points]

Use `country_plot` to plot three versions of the plot colored by `Fertility2020` column. As the pallete, use the following three:

* `flare`, 
* `rocket` with order of colors [reversed](https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.Colormap.html#matplotlib.colors.Colormap.reversed) and 
* a [light palette](https://seaborn.pydata.org/generated/seaborn.light_palette.html#seaborn.light_palette) using blue as the basic color. 

When creating palettes, do not forget argument `as_cmap=True`. Mention the palette in the plot title.

You may notice that some countries are hard to see. You will discuss this in the quiz.

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8,15), layout='tight')

### TASK 2 START
# country_plot(axes[0],...)
# axes[0].set_title(...)
# ...
### TASK 2 END

pass

## Task 2b: quantitative palettes - discussion [1 point]

The [tutorial](https://seaborn.pydata.org/tutorial/color_palettes.html) mentions that `rocket` is better for heatmaps rather than scatter plots. Its use made one country in the plot almost invisible. Answer the following questions:

### TASK 2b START

Which country is hardest to see in the plot? What is the reason? 

Is this also problem for the light palette?

### TASK 2b END

## Task 3 [3 points]

Divergent palette `vlag` uses red colors for high values, blue for low values and white for the middle value. If we would use `country_plot` with this palette and column `GDPRelChange`, we would expect that white is used for zero. However, since the minimum is -0.95 and the maximum 0.36, white will in fact correspond to the middle between these two extremes, which is about -0.30. You can see this in the top plot below.

Write a function `country_plot_divergent`, which will be very similar to `country_plot` (copy the code of `country_plot` and modify it), but it will ensure that the middle color of the palette corresponds to zero. This is achieved by argument `hue_norm` in [scatterplot](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) function, which can take minimum and maximum values used in mapping values to the colormap. For `GDPRelChange`, we would like to set `hue_norm=(-0.95,0.95)`. However, you should implement the function so that it works for any numerical column. For example, if the values of the column are from -1 to 2, set `hue_norm=(-2,2)`. If all values are positive, for example from 10 to 20, set `hue_norm=(-20, 20)` and only red colors will be used. Similarly for negative values only blue colors are used.

Also include parameters `edgecolor="black", linewidth=0.3` in `sns.scatterplot` to solve the problem with invisible countries.

Your function should return the limits used in `hue_norm` as the return value, which is then displayed in plot titles in the code below.

In [ ]:
def country_plot_divergent(axes, colorby, palette):
  ### TASK 3 START
  # modified code from country_plot
  # ...
  # sns.scatterplot(... , hue_norm=(..., ...), edgecolor="black", linewidth=0.3)
  # ...
  # return (..., ...)
  ### TASK 3 END

The code below first uses the original function with `vlag`, then calls your function with `GDPRelChange`, where white should correspond to 0. Then it calls your function with  `Fertility2020` where only red colors should be used. Finally it creates modified versions of columns `GDPRelChange` and `Fertility2020` with all values multipled by -1 and displays these to check the correctness of your code for various cases.

In [ ]:
figure, axes = plt.subplots(5, 1, figsize=(8,25))
figure.subplots_adjust(hspace=0.25)

country_plot(axes[0], 'GDPRelChange', sns.color_palette("vlag", as_cmap=True))
axes[0].set_title('GDP rel. change using default hue_norm')

limits = country_plot_divergent(axes[1], 'GDPRelChange', 
                                sns.color_palette("vlag", as_cmap=True))
axes[1].set_title(f'GDP rel. change using hue_norm=({limits[0]:.3f}, {limits[1]:.3f})')

limits = country_plot_divergent(axes[2], 'Fertility2020', 
                                sns.color_palette("vlag", as_cmap=True))
axes[2].set_title(f'Fertility 2020 using hue_norm=({limits[0]:.3f}, {limits[1]:.3f})')

selection['MinusGDPRelChange'] = -selection['GDPRelChange']
limits = country_plot_divergent(axes[3], 'MinusGDPRelChange', 
                                sns.color_palette("vlag", as_cmap=True))
axes[3].set_title(f'Minus GDPRelChange using hue_norm=({limits[0]:.3f}, {limits[1]:.3f})')

selection['MinusFertility2020'] = -selection['Fertility2020']
limits = country_plot_divergent(axes[4], 'MinusFertility2020', 
                                sns.color_palette("vlag", as_cmap=True))
axes[4].set_title(f'Minus Fertility 2020 using hue_norm=({limits[0]:.3f}, {limits[1]:.3f})')

os.makedirs("img", exist_ok=True)  # make a folder for images
figure.savefig("img/HW08-t3.png", bbox_inches='tight')  

pass